# 1. Import Libraries, Configuration Setup, and Load the Dataset

**Experiment:** `06_Semi_confirmatory` — anchored plus free attention factors in the `05_` fine-tuned family.

**Anchored** factors are `05_`'s aspect representations: one token-attention head per predefined aspect, each supervised by its three-class aspect label. **Free** factors are the same attention heads without an aspect head; they are shaped only by the holistic loss and by an attention-overlap (cosine) penalty that pushes them away from the anchored factors and from each other. They are unconstrained residual representation factors, not discovered latent aspects, until their loadings recur across seeds. A review-specific fusion combines all factors into the holistic prediction.

`ANCHORED_ASPECTS` and `FREE_FACTORS` in 1.3 select the design: four anchored and zero free is the confirmatory model (`05_` `representation_only`); four anchored plus *m* free is semi-confirmatory; zero anchored plus *K* free is the exploratory model of the same family. `RESIDUAL_CLS` optionally adds the direct CLS route of `05_` `representation_residual`. Each setting writes to its own output folder.

All conditions use review text only, the frozen label release and split, and the existing 04 loss/search/selection conventions. Special tokens and padding cannot receive attention. For every factor the audit file records the most attended tokens and spans; every evaluated partition gets a factor-loading table, the attention mass each factor places on the GPT evidence spans of each aspect relative to chance, which is how free factors are related to the known aspects after fitting. Loadings are descriptive correspondence, not validated new aspects.

**This notebook is fixed to 4 anchored + 0 free factors.** The sibling notebook in this folder covers the other design; each writes to its own `outputs/semi_<anchored>anchored_<free>free_v2` folder.


## 1.1 Install \& Import Libraries

In [1]:
# Run before imports in a fresh Colab runtime. PyTorch is supplied by Colab.
%pip -q install transformers==4.57.6 scikit-learn pandas matplotlib seaborn nltk lightgbm statsmodels tqdm joblib


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 212.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 203.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.27.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
# No additional install is needed here; packages are installed in the cell above.


In [3]:
# No additional install is needed here; packages are installed in the cell above.


In [4]:
# No additional install is needed here; packages are installed in the cell above.


In [5]:
import os
import re
import gc
import json
import math
import random
import hashlib
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score, cohen_kappa_score, classification_report, confusion_matrix

import time
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

def aux_hash(path, algorithm="sha256"):
    h = hashlib.new(algorithm)
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1 << 24), b""):
            h.update(chunk)
    return h.hexdigest()

def aux_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")
    tmp.replace(path)

def aux_validate_config(cfg):
    aspects = tuple(cfg["aspects"])
    if len(set(aspects)) != len(aspects) or any(a not in AUX_ASPECTS for a in aspects):
        raise ValueError("Unknown or repeated aspect.")
    if cfg["target"] not in ("original", "holistic", "aspects"):
        raise ValueError("Unknown training target.")
    if cfg["target"] == "original" and aspects:
        raise ValueError("Original baseline has no auxiliary heads.")
    if cfg["target"] == "aspects" and aspects != AUX_ASPECTS:
        raise ValueError("Standalone full-aspects condition must predict all four aspects.")
    if not math.isfinite(cfg["aux_strength"]) or cfg["aux_strength"] < 0:
        raise ValueError("Auxiliary strength must be finite and non-negative.")
    if cfg["target"] == "holistic" and aspects and cfg["aux_strength"] <= 0:
        raise ValueError("Joint condition needs positive auxiliary strength.")
    if cfg["n_trials"] < 1 or cfg["batch_size"] < 1 or cfg["max_length"] < 1:
        raise ValueError("Training sizes must be positive.")
    if not cfg["backbones"] or any(b not in AUX_BACKBONES for b in cfg["backbones"]):
        raise ValueError("Select ALBERT and/or BioBERT.")
    # Semi-confirmatory model: anchored (supervised) plus free (unsupervised) factors.
    if cfg["target"] != "holistic":
        raise ValueError("The semi-confirmatory model predicts the holistic label.")
    if not isinstance(cfg["free_factors"], int) or cfg["free_factors"] < 0:
        raise ValueError("free_factors must be a non-negative integer.")
    if not aspects and cfg["free_factors"] == 0:
        raise ValueError("Select at least one anchored aspect or one free factor.")
    if cfg["attention_size"] < 1:
        raise ValueError("Attention size must be positive.")
    if not math.isfinite(cfg["orthogonality"]) or cfg["orthogonality"] < 0:
        raise ValueError("Orthogonality weight must be finite and non-negative.")
    if not isinstance(cfg["residual_cls"], bool):
        raise ValueError("residual_cls must be True or False.")


def require_finite_tensor(value, description):
    if not torch.isfinite(value).all():
        raise FloatingPointError(f"Non-finite {description}; no checkpoint or prediction will be selected.")

def require_finite_model(model):
    for name, value in model.state_dict().items():
        if value.is_floating_point():
            require_finite_tensor(value, f"model state {name}")

def require_probabilities(values, n_classes):
    values = np.asarray(values)
    if (values.ndim != 2 or values.shape[1] != n_classes or not np.isfinite(values).all()
            or (values < 0).any() or (values > 1).any()
            or not np.allclose(values.sum(axis=1), 1.0, atol=1e-5, rtol=1e-5)):
        raise ValueError("Invalid probability matrix.")
    return values

def checked_optimizer_step(loss, model, optimizer, scaler, max_norm, scheduler=None):
    require_finite_tensor(loss, "training loss")
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    grad_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(), max_norm, error_if_nonfinite=not scaler.is_enabled())
    previous_scale = scaler.get_scale()
    scaler.step(optimizer)
    scaler.update()
    updated = scaler.get_scale() >= previous_scale
    if updated and not torch.isfinite(grad_norm):
        raise FloatingPointError("Non-finite gradient norm without a skipped optimizer update.")
    if updated and scheduler is not None:
        scheduler.step()
    return updated


def aux_model_contract():
    """Configuration-cell constants that affect model/data meaning are recorded explicitly."""
    keys = ("AUX_BACKBONES", "AUX_ASPECTS", "AUX_STATES", "AUX_OVERALL", "AUX_SOURCE_STATES",
            "AUX_STATE_TO_CLASS", "AUX_RELEASE", "EVIDENCE_CLASSES", "EVIDENCE_IGNORE",
            "EVIDENCE_ALIGNMENT_VERSION", "UPSTREAM_MANIFEST_SHA256", "INPUT_CONDITIONS")
    # JSON normalization avoids tuple/list mismatches when a completed identity is read again.
    return json.loads(json.dumps({key: globals()[key] for key in keys if key in globals()}, sort_keys=True))


## 1.2 Seed \& Device

In [6]:

def aux_seed(seed):
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

def aux_worker_seed(worker_id):
    seed = torch.initial_seed() % 2**32
    np.random.seed(seed)
    random.seed(seed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}; select the experiment seed in 1.3.")
# Trial i uses RUN_SEED + i; the saved data split is independent of training seeds.


Device: cuda; select the experiment seed in 1.3.


## 1.3 Configuration

In [7]:
RUN_SEED = 2025
SEED = RUN_SEED
aux_seed(RUN_SEED)

PROJECT_ROOT = Path("/content/drive/MyDrive/NLP_Projects/03_DrugReview")
EXPERIMENT_FOLDER = '06_Semi_confirmatory'
TRAINING_TARGET = 'holistic'
ANCHORED_ASPECTS = ['efficacy', 'safety', 'burden', 'cost']   # supervised factors; [] for the exploratory model
ASPECT_NAMES = ANCHORED_ASPECTS
USE_METADATA = False

# Factor design; see section 3.3.
FREE_FACTORS = 0                # fixed for this notebook (0 = confirmatory); the sibling notebook covers the other design
ORTHOGONALITY = 0.1             # weight of the attention-orthogonality penalty on pairs involving a free factor
RESIDUAL_CLS = False            # True adds the direct CLS route of 05_ representation_residual
ATTENTION_SIZE = 256            # hidden size of the token-level and factor-level attention scorers

# Existing DrugReview settings; unchanged from 03_/04_/05_.
N_ITERATIONS = 10
MAX_TOKEN_LENGTH = 200
BATCH_SIZE = 128
PATIENCE = 3
NUM_WORKERS = 2
AUXILIARY_STRENGTH = 1.0
BACKBONES = ['albert', 'biobert']

AUX_CONFIG = {
    "folder": EXPERIMENT_FOLDER, "target": TRAINING_TARGET,
    "aspects": ASPECT_NAMES, "augmentation": USE_METADATA,
    "aux_strength": AUXILIARY_STRENGTH, "n_trials": N_ITERATIONS,
    "seed": RUN_SEED, "max_length": MAX_TOKEN_LENGTH,
    "batch_size": BATCH_SIZE, "patience": PATIENCE,
    "num_workers": NUM_WORKERS, "backbones": BACKBONES,
    "free_factors": FREE_FACTORS, "orthogonality": ORTHOGONALITY,
    "residual_cls": RESIDUAL_CLS, "attention_size": ATTENTION_SIZE,
}

FACTOR_NAMES = list(ANCHORED_ASPECTS) + [f"free_{i + 1}" for i in range(FREE_FACTORS)]
AUX_VERSION = f"semi_{len(ANCHORED_ASPECTS)}anchored_{FREE_FACTORS}free{'_cls' if RESIDUAL_CLS else ''}_v2"
AUX_ASPECTS = ("efficacy", "safety", "burden", "cost")
# The immutable GPT release keeps all five source states. Only supervision is collapsed.
AUX_SOURCE_STATES = ("NEGATIVE", "POSITIVE", "MIXED", "UNMENTIONED", "UNCLEAR")
AUX_STATES = ("NEGATIVE", "NEUTRAL", "POSITIVE")
AUX_STATE_TO_CLASS = {
    "NEGATIVE": "NEGATIVE", "POSITIVE": "POSITIVE",
    "MIXED": "NEUTRAL", "UNMENTIONED": "NEUTRAL", "UNCLEAR": "NEUTRAL",
}
# NEUTRAL is the pooled human-rubric class; it need not express a neutral attitude.
AUX_OVERALL = ("VERY_NEGATIVE", "NEGATIVE", "NEUTRAL", "POSITIVE", "VERY_POSITIVE")
AUX_RELEASE = {
    "protocol_id": "v2_ef356fdf87",
    "label_file_md5": "c22f6c05633214014db47c692e045ea1",
    "split_fingerprint": "038e1d09a2818dfcf6a91dce49e35a22",
    "n_eligible": 102061,
}
AUX_BACKBONES = {"albert": "albert-base-v2", "biobert": "dmis-lab/biobert-base-cased-v1.2"}

# Implementation fingerprint: sha256 of code cells except this editable configuration
# cell (index 10), joined with a single newline. Configuration and model-contract
# constants are saved separately in experiment_identity.json. Configuration-only
# ablations need no manual fingerprint changes; implementation edits do.
LEGACY_AUX_RUNTIME_SHA256S = []
AUX_RUNTIME_SHA256 = "dbb217247488e4df04b93f33afa67e7b2aca00b84ddf61b59ae7cd8c7fe9c1ec"
OUTPUT_PATH = PROJECT_ROOT / EXPERIMENT_FOLDER / "outputs" / AUX_VERSION
ALBERT_MODEL_NAME = AUX_BACKBONES["albert"]
BIOBERT_MODEL_NAME = AUX_BACKBONES["biobert"]
print("Experiment:", EXPERIMENT_FOLDER, "| anchored:", ANCHORED_ASPECTS, "| free factors:", FREE_FACTORS, "| residual CLS:", RESIDUAL_CLS, "| trials per model:", N_ITERATIONS)
print("Outputs:", OUTPUT_PATH)


Experiment: 06_Semi_confirmatory | anchored: ['efficacy', 'safety', 'burden', 'cost'] | free factors: 0 | residual CLS: False | trials per model: 10
Outputs: /content/drive/MyDrive/NLP_Projects/03_DrugReview/06_Semi_confirmatory/outputs/semi_4anchored_0free_v2


## 1.4 Mount drive & Load the data

In [8]:
if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

def load_label_release(project_root, cfg):
    aux_validate_config(cfg)
    project_root = Path(project_root)
    source = project_root / "00_labelling/output_v2/v2_ef356fdf87/drugscom_plus_ai_labels_v2_ef356fdf87.csv"
    split_path = project_root / "00_split/data/DrugReview_split.csv"
    if aux_hash(source, "md5") != AUX_RELEASE["label_file_md5"]:
        raise ValueError("Label release changed. Restore the pinned release; do not silently relabel.")
    df = pd.read_csv(source, index_col=0).reset_index()
    split = pd.read_csv(split_path)
    required = {"uniqueID", "review", "eligible_both_streams", "rating", "ai_sent5_hard"}
    required |= {f"ai_{a}_state" for a in AUX_ASPECTS}
    if not required.issubset(df.columns):
        raise ValueError(f"Missing release columns: {required - set(df.columns)}")
    if not {"uniqueID", "split", "narrative_group", "is_audit_group"}.issubset(split.columns):
        raise ValueError("Incomplete split manifest.")
    for frame in (df, split):
        if frame.uniqueID.isna().any() or not frame.uniqueID.is_unique:
            raise ValueError("Missing or duplicate review IDs.")
    if set(df.uniqueID) != set(split.uniqueID):
        raise ValueError("Release and manifest ID sets differ.")
    fingerprint = hashlib.md5("|".join(
        f"{u}:{s}" for u, s in sorted(zip(split.uniqueID.astype(str), split["split"]))
    ).encode()).hexdigest()
    if fingerprint != AUX_RELEASE["split_fingerprint"]:
        raise ValueError("Frozen split changed.")
    if not split["split"].isin(["train", "val", "test", "audit"]).all():
        raise ValueError("Unknown split assignment.")
    if split.narrative_group.isna().any() or (split.groupby("narrative_group")["split"].nunique() > 1).any():
        raise ValueError("Missing narrative groups or narratives cross splits.")
    if not split.is_audit_group.isin([True, False]).all() or not (split.is_audit_group == split["split"].eq("audit")).all():
        raise ValueError("Audit reservation is inconsistent.")
    if not df.eligible_both_streams.isin([True, False]).all():
        raise ValueError("Invalid cohort eligibility flags.")
    df = df.loc[df.eligible_both_streams].copy()
    if len(df) != AUX_RELEASE["n_eligible"]:
        raise ValueError("Eligible cohort changed.")
    df = df.drop(columns=[c for c in split if c != "uniqueID" and c in df])
    df = df.merge(split, on="uniqueID", validate="one_to_one").sort_values("uniqueID").reset_index(drop=True)
    return df, split, source, split_path

df, split, source, split_path = load_label_release(PROJECT_ROOT, AUX_CONFIG)
print(f"Loaded {len(df):,} eligible reviews from {source}")


Mounted at /content/drive
Loaded 102,061 eligible reviews from /content/drive/MyDrive/NLP_Projects/03_DrugReview/00_labelling/output_v2/v2_ef356fdf87/drugscom_plus_ai_labels_v2_ef356fdf87.csv


In [9]:
# The loader has checked review IDs, the frozen label file and the saved split.
print("Saved partitions:", df["split"].value_counts().to_dict())


Saved partitions: {'train': 60950, 'test': 20298, 'val': 20256, 'audit': 557}


In [10]:
# Keep the existing review order and split; do not create a new random split.
print("Review IDs are sorted; training seeds do not change partition membership.")


Review IDs are sorted; training seeds do not change partition membership.


# 2. Data Exploration \& Master Data Preparation

## 2.1 Master Data Preparation

### 2.1.1 Text cleaning

In [11]:
def aux_clean_text(text):
    # Same cleaning as the existing hard-label and aspect notebooks.
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"http\S+", " urltoken ", text)
    text = re.sub(r"@\w+", " usertoken ", text)
    text = re.sub(r"#(\w+)", r" hashtag_\1 ", text)
    text = re.sub(r"[^\w\s]", "", text)
    return re.sub(r"\s+", " ", text).strip()

text = df.review
if AUX_CONFIG["augmentation"]:
    # Reuse the exact metadata fields and prefix order from 03/05 _aug.
    for col in ("condition_norm", "ingredient_set_key"):
        df[col] = df[col].fillna("").astype(str).str.lower().str.strip().str.replace(r"\s+", " ", regex=True)
    df["n_ingredients"] = pd.to_numeric(df.n_ingredients, errors="coerce").fillna(0).astype(int)
    text = df.apply(lambda r: f"[COND] {r.condition_norm} [ING] {r.ingredient_set_key} [N_ING] {r.n_ingredients} [TEXT] {r.review}", axis=1)
df["model_text"] = text.apply(aux_clean_text)

print("Input:", "review plus metadata" if USE_METADATA else "review text only")


Input: review text only


### 2.1.2 Validate & encode aspect labels

NEGATIVE = 0, NEUTRAL = 1, POSITIVE = 2. Source states MIXED, UNMENTIONED and UNCLEAR are grouped into NEUTRAL. The source columns are preserved. The holistic target, when used, keeps its five classes.


In [12]:
# Validate all targets, so baseline and joint conditions have the same cohort.
if not df.ai_sent5_hard.isin(AUX_OVERALL).all():
    raise ValueError("Invalid holistic label; never replace failed labels with neutral.")
rating = pd.to_numeric(df.rating, errors="coerce").round()
if not rating.between(1, 10).all():
    raise ValueError("Invalid rating would change the common cohort.")
df["original_id"] = ((rating.astype(int) - 1) // 2).astype(int)
df["holistic_id"] = df.ai_sent5_hard.map({x: i for i, x in enumerate(AUX_OVERALL)}).astype(int)
for aspect in AUX_ASPECTS:
    col = f"ai_{aspect}_state"
    if not df[col].isin(AUX_SOURCE_STATES).all():
        raise ValueError(f"Invalid five-state label: {col}")
    df[f"{aspect}_id"] = df[col].map(AUX_STATE_TO_CLASS).map(
        {x: i for i, x in enumerate(AUX_STATES)}).astype(int)

for aspect in ASPECT_NAMES:
    print(aspect, df[f"{aspect}_id"].value_counts().sort_index().to_dict())


efficacy {0: 19761, 1: 17551, 2: 64749}
safety {0: 51921, 1: 34895, 2: 15245}
burden {0: 40844, 1: 51341, 2: 9876}
cost {0: 4689, 1: 94905, 2: 2467}


## 2.3 Master Data Split

Use the existing train, validation, test and reserved audit assignments from `00_split`. Training and parameter selection use only train and validation rows.


In [13]:
frames = {s: df.loc[df["split"].eq(s)].copy().reset_index(drop=True) for s in ("train", "val", "test", "audit")}
if any(len(frame) == 0 for frame in frames.values()):
    raise ValueError("Each frozen partition must be nonempty.")
identity = {
    "release": AUX_RELEASE, "split_file_sha256": aux_hash(split_path),
    "cohort_sha256": hashlib.sha256("|".join(df.uniqueID.astype(str)).encode()).hexdigest(),
    "split_counts": {s: len(f) for s, f in frames.items()},
    "text_sha256": hashlib.sha256("\n".join(df.model_text).encode()).hexdigest(),
}

print("Partitions:", identity["split_counts"])


Partitions: {'train': 60950, 'val': 20256, 'test': 20298, 'audit': 557}


# 3. ALBERT \& BioBERT Models (Semi-confirmatory)


## 3.1 Shared helper functions

Anchored factors have three-class aspect heads; free factors have none. The holistic head (five classes) reads the fused factors, and optionally CLS. Model probabilities come from the trained classifier, not from the GPT annotation API. Every prediction file carries the fusion weight of each factor; the factor-loading table relates each factor's token attention to the GPT evidence spans of the four aspects; the audit file adds top tokens and character spans per factor.


In [14]:
def aux_task_labels(frame, cfg):
    labels = {a: frame[f"{a}_id"].to_numpy() for a in cfg["aspects"]}
    if cfg["target"] != "aspects":
        labels["overall"] = frame[f"{cfg['target']}_id"].to_numpy()
    return labels

def aux_criteria(train_labels, device):
    # Retain balanced CE within each task, calculated on TRAIN only.
    criteria, weights = {}, {}
    for task, values in train_labels.items():
        n_classes = len(AUX_OVERALL) if task == "overall" else len(AUX_STATES)
        values = np.asarray(values)
        if values.ndim != 1 or len(values) == 0 or not np.issubdtype(values.dtype, np.integer) or not ((values >= 0) & (values < n_classes)).all():
            raise ValueError(f"Invalid {n_classes}-class training targets: {task}")
        counts = np.bincount(values, minlength=n_classes)
        seen = counts > 0
        weight = np.zeros(n_classes, dtype=np.float32)
        weight[seen] = len(values) / (seen.sum() * counts[seen])
        weights[task] = weight.tolist()
        criteria[task] = nn.CrossEntropyLoss(weight=torch.tensor(weight, device=device))
    return criteria, weights

def aux_loss(logits, targets, criteria, strength, orthogonality=0.0, penalty=None):
    terms = {k: criteria[k](logits[k].float(), targets[k]) for k in logits if k in criteria}
    aspect_terms = [v for k, v in terms.items() if k != "overall"]
    total = terms["overall"] if "overall" in terms else torch.stack(aspect_terms).mean()
    if "overall" in terms and aspect_terms:
        total = total + strength * torch.stack(aspect_terms).mean()
    if penalty is not None and orthogonality > 0:
        terms["orthogonality"] = penalty
        total = total + orthogonality * penalty
    return total, terms

def aux_metrics(y, pred, ordinal=False):
    # Aspect macro F1 averages three classes; overall metrics retain five.
    class_ids = range(len(AUX_OVERALL) if ordinal else len(AUX_STATES))
    out = {
        "n": len(y), "accuracy": float(accuracy_score(y, pred)),
        "weighted_f1": float(f1_score(y, pred, labels=class_ids, average="weighted", zero_division=0)),
        "macro_f1": float(f1_score(y, pred, labels=class_ids, average="macro", zero_division=0)),
    }
    if ordinal:
        qwk = float(cohen_kappa_score(y, pred, labels=range(5), weights="quadratic"))
        out.update(mae=float(np.mean(np.abs(np.asarray(y) - pred))), qwk=qwk if np.isfinite(qwk) else None)
    return out

def aux_predict(model, loader, device, keep_tokens=False):
    """Predict every task; collect fusion weights, factor loadings on evidence spans and, on request, token attention."""
    model.eval()
    probs, labels = {}, {}
    fusion, token_attention, input_ids, eligible_masks, offsets = [], [], [], [], []
    n_factors = len(FACTOR_NAMES)
    mass = np.zeros((n_factors, len(AUX_ASPECTS)))       # attention mass on each aspect's evidence tokens
    expected = np.zeros(len(AUX_ASPECTS))                 # chance mass: evidence tokens / eligible tokens
    counted = np.zeros(len(AUX_ASPECTS))
    with torch.no_grad():
        for batch in loader:
            logits, extras = model(batch["input_ids"].to(device), batch["attention_mask"].to(device),
                special_tokens_mask=batch["special_tokens_mask"].to(device), return_extras=True)
            for task, values in logits.items():
                require_finite_tensor(values, f"{task} prediction logits")
                probabilities = values.float().softmax(-1).cpu().numpy()
                require_probabilities(probabilities, values.shape[-1])
                probs.setdefault(task, []).append(probabilities)
                labels.setdefault(task, []).append(batch["targets"][task].numpy())
            require_finite_tensor(extras["fusion"], "factor fusion weights")
            fusion.append(extras["fusion"].float().cpu().numpy())
            attention = extras["token_attention"].float().cpu()                                  # (B, F, T)
            require_finite_tensor(attention, "token attention")
            eligible = (batch["attention_mask"].bool() & ~batch["special_tokens_mask"].bool())
            if "evidence_tokens" in batch:
                evidence = (batch["evidence_tokens"].bool() & eligible.unsqueeze(1)).float()      # (B, A, T)
                has_evidence = evidence.sum(-1) > 0                                              # (B, A)
                batch_mass = torch.einsum("bft,bat->bfa", attention, evidence)                   # (B, F, A)
                mass += (batch_mass * has_evidence.unsqueeze(1)).sum(0).numpy()
                expected += ((evidence.sum(-1) / eligible.float().sum(-1).unsqueeze(1).clamp(min=1)) * has_evidence).sum(0).numpy()
                counted += has_evidence.sum(0).numpy()
            if keep_tokens:
                token_attention.append(attention.numpy())
                input_ids.append(batch["input_ids"].numpy())
                eligible_masks.append(eligible.numpy())
                offsets.append(batch["offset_mapping"].numpy())
    extras = {
        "fusion": np.concatenate(fusion),
        "token_attention": np.concatenate(token_attention) if token_attention else None,
        "input_ids": np.concatenate(input_ids) if input_ids else None,
        "eligible_mask": np.concatenate(eligible_masks) if eligible_masks else None,
        "offset_mapping": np.concatenate(offsets) if offsets else None,
        "loadings": {"mass": mass, "expected": expected, "counted": counted},
    }
    return {k: np.concatenate(v) for k, v in probs.items()}, {k: np.concatenate(v) for k, v in labels.items()}, extras

def aux_selection(probs, labels):
    for task, p in probs.items():
        require_probabilities(p, 5 if task == "overall" else 3)
    if "overall" in probs:
        return float(f1_score(labels["overall"], probs["overall"].argmax(1), average="weighted", zero_division=0))
    return float(np.mean([f1_score(labels[k], p.argmax(1), average="weighted", zero_division=0) for k, p in probs.items()]))

def aux_trial_plan(cfg):
    rng = np.random.RandomState(cfg["seed"])
    return [{"lr": float(10**rng.uniform(np.log10(1e-5), np.log10(3e-5))),
             "epochs": int(rng.randint(4, 10)), "dropout": float(rng.uniform(0.0, 0.25)),
             "seed": cfg["seed"] + i} for i in range(cfg["n_trials"])]

def aux_top_positions(token_attention, input_ids, tokenizer, eligible_mask, k=8):
    weights, ids = np.asarray(token_attention), np.asarray(input_ids)
    eligible = np.asarray(eligible_mask, dtype=bool) & ~np.isin(ids, tokenizer.all_special_ids)
    if weights.shape != ids.shape or eligible.shape != ids.shape or not np.isfinite(weights).all():
        raise ValueError("Malformed token explanation arrays.")
    positions = np.flatnonzero(eligible)
    return positions[np.argsort(-weights[positions], kind="stable")[:max(0, k)]]

def aux_top_tokens(token_attention, input_ids, tokenizer, eligible_mask, k=8):
    order = aux_top_positions(token_attention, input_ids, tokenizer, eligible_mask, k)
    tokens = tokenizer.convert_ids_to_tokens([int(input_ids[i]) for i in order])
    return "|".join(f"{t}:{token_attention[i]:.3f}" for t, i in zip(tokens, order))

def aux_top_spans(token_attention, input_ids, tokenizer, eligible_mask, offsets, k=8):
    order = aux_top_positions(token_attention, input_ids, tokenizer, eligible_mask, k)
    tokens = tokenizer.convert_ids_to_tokens([int(input_ids[i]) for i in order])
    return json.dumps([{"token": t, "start": int(offsets[i][0]), "end": int(offsets[i][1]),
                        "attention": float(token_attention[i])} for t, i in zip(tokens, order)], allow_nan=False)

def aux_evidence_tokens(encoded, frame):
    """Token mask (N, aspects, T) of the GPT evidence quotes located verbatim in the cleaned text; evaluation only."""
    offsets = encoded["offset_mapping"]
    starts, ends = offsets[:, :, 0], offsets[:, :, 1]
    real = (ends > starts) & ~encoded["special_tokens_mask"].bool()
    mask = torch.zeros((len(frame), len(AUX_ASPECTS), offsets.shape[1]), dtype=torch.uint8)
    visible_end = torch.where(real, ends, torch.zeros_like(ends)).max(dim=1).values      # last character the encoder sees
    counts = {"total": 0, "unmatched": 0, "ambiguous": 0, "truncated": 0, "used": 0}
    texts = frame.model_text.tolist()
    for a, aspect in enumerate(AUX_ASPECTS):
        for i, evidence_json in enumerate(frame[f"ai_{aspect}_evidence"].tolist()):
            try:
                quotes = json.loads(evidence_json) if isinstance(evidence_json, str) else []
            except ValueError:
                quotes = []
            for quote, _direction in quotes:
                counts["total"] += 1
                cleaned = aux_clean_text(quote)
                occurrences = texts[i].count(cleaned) if cleaned else 0
                if occurrences == 0:
                    counts["unmatched"] += 1
                    continue
                if occurrences > 1:                      # cannot tell which occurrence was meant
                    counts["ambiguous"] += 1
                    continue
                position = texts[i].find(cleaned)
                if position + len(cleaned) > int(visible_end[i]):   # quote runs past the token limit
                    counts["truncated"] += 1
                    continue
                counts["used"] += 1
                span = (starts[i] < position + len(cleaned)) & (ends[i] > position) & real[i]
                mask[i, a] |= span.to(torch.uint8)
    print("Evidence quotes:", counts, "(only unique, fully visible quotes define evidence tokens)")
    return mask


def aux_export(model, loader, frame, cfg, model_name, split_name, output, device, tokenizer=None):
    keep_tokens = split_name == "audit" and tokenizer is not None
    probs, labels, extras = aux_predict(model, loader, device, keep_tokens=keep_tokens)
    audit_columns = [c for c in ("is_human_annotated", "is_audit_group", "audit_stratum") if c in frame]
    result = frame[["uniqueID", "narrative_group", "split"] + audit_columns].copy()
    metrics = []
    design = f"{len(cfg['aspects'])}anchored_{cfg['free_factors']}free{'_cls' if cfg['residual_cls'] else ''}"
    for task, p in probs.items():
        names = AUX_OVERALL if task == "overall" else AUX_STATES
        if p.ndim != 2 or p.shape != (len(frame), len(names)):
            raise ValueError(f"Unexpected probability shape for {task}: {p.shape}")
        require_probabilities(p, len(names))
        pred, y = p.argmax(1), labels[task]
        result[f"{task}_true_id"] = y
        result[f"{task}_pred_id"] = pred
        result[f"{task}_pred"] = np.asarray(names)[pred]
        for k, name in enumerate(names):
            result[f"{task}_p_{name}"] = p[:, k]
        row = {"model": model_name, "variant": design, "task": task, "partition": split_name, "reference_source": "rating" if cfg["target"] == "original" and task == "overall" else "GPT-4o-mini", "human_agreement": False, **aux_metrics(y, pred, task == "overall")}
        metrics.append(row)
        if split_name == "test":
            pd.DataFrame(classification_report(y, pred, labels=range(len(names)), target_names=names,
                output_dict=True, zero_division=0)).T.to_csv(output / f"{model_name}_{task}_class_report.csv")
            pd.DataFrame(confusion_matrix(y, pred, labels=range(len(names))), index=names, columns=names).to_csv(
                output / f"{model_name}_{task}_confusion.csv")
        if task != "overall":
            # Native three-class predictions; these compatibility columns are exact aliases.
            result[f"{task}_true_3class_id"] = y
            result[f"{task}_pred_3class_id"] = pred
            source_col = f"ai_{task}_state"
            if source_col in frame:
                result[f"{task}_source_state"] = frame[source_col].to_numpy()
                result[f"{task}_is_mentioned"] = frame[source_col].ne("UNMENTIONED").to_numpy()
    # Review-specific fusion weights over all factors (anchored first, then free).
    fusion = require_probabilities(extras["fusion"], len(FACTOR_NAMES))
    if fusion.shape[0] != len(frame):
        raise ValueError(f"Unexpected fusion shape: {fusion.shape}")
    for k, factor in enumerate(FACTOR_NAMES):
        result[f"fusion_w_{factor}"] = fusion[:, k]
    if split_name == "test":
        summary = result.groupby("overall_true_id")[[f"fusion_w_{f}" for f in FACTOR_NAMES]].mean()
        summary.index = np.asarray(AUX_OVERALL)[summary.index.to_numpy()]
        summary.to_csv(output / f"{model_name}_fusion_by_holistic_class.csv")
    # Factor loadings: attention mass on each aspect's evidence tokens, relative to chance.
    loadings, rows = extras["loadings"], []
    for k, factor in enumerate(FACTOR_NAMES):
        for a, aspect in enumerate(AUX_ASPECTS):
            n = loadings["counted"][a]
            if n == 0:
                continue
            mean_mass, chance = loadings["mass"][k, a] / n, loadings["expected"][a] / n
            rows.append({"model": model_name, "partition": split_name, "factor": factor, "aspect": aspect,
                         "n_reviews_with_evidence": int(n), "attention_on_evidence": mean_mass,
                         "chance": chance, "lift": mean_mass / chance if chance > 0 else np.nan})
    if rows:
        pd.DataFrame(rows).to_csv(output / f"{model_name}_{split_name}_factor_loadings.csv", index=False)
    # Audit partition only: the most attended tokens and spans per factor.
    if keep_tokens and extras["token_attention"] is not None:
        attention, ids = extras["token_attention"], extras["input_ids"]
        eligible, offsets = extras["eligible_mask"], extras["offset_mapping"]
        result["model_text"] = frame.model_text.to_numpy()
        if attention.shape[:2] != (len(frame), len(FACTOR_NAMES)) or ids.shape[0] != len(frame):
            raise ValueError("Unexpected token attention shape.")
        for k, factor in enumerate(FACTOR_NAMES):
            result[f"{factor}_top_tokens"] = [aux_top_tokens(attention[i, k], ids[i], tokenizer, eligible[i]) for i in range(len(frame))]
            result[f"{factor}_top_spans_json"] = [aux_top_spans(attention[i, k], ids[i], tokenizer, eligible[i], offsets[i]) for i in range(len(frame))]
    result.to_csv(output / f"{model_name}_{split_name}_predictions.csv", index=False)
    return metrics

def prepare_experiment(project_root, cfg, frames, identity):
    """Check saved settings and prepare the output folder and training losses."""
    import transformers
    import sklearn
    output = Path(project_root) / cfg["folder"] / "outputs" / AUX_VERSION
    output.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    plan = aux_trial_plan(cfg)
    comparable_cfg = {k: v for k, v in cfg.items() if k not in ("folder", "backbones", "notebook")}
    signature = {"version": AUX_VERSION, "runtime_sha256": AUX_RUNTIME_SHA256, "model_contract": aux_model_contract(),
        "config": comparable_cfg, "data": identity, "trials": plan,
        "aspect_schema": {"classes": list(AUX_STATES), "source_mapping": AUX_STATE_TO_CLASS},
        "factors": {"anchored": list(cfg["aspects"]), "free": cfg["free_factors"], "names": list(FACTOR_NAMES),
                    "orthogonality": cfg["orthogonality"], "residual_cls": cfg["residual_cls"], "attention_size": cfg["attention_size"]},
        "python": platform.python_version(), "torch": torch.__version__,
        "transformers": transformers.__version__, "sklearn": sklearn.__version__}
    # Never silently load stale checkpoints when data, settings, code or libraries differ.
    identity_path = output / "experiment_identity.json"
    if identity_path.exists():
        saved_signature = json.loads(identity_path.read_text())
        expected_signature = dict(signature)
        # Accept the previous cell layout only when every data/training setting matches.
        # Keep the original saved identity; do not relabel an old run as a new one.
        if saved_signature.get("runtime_sha256") in LEGACY_AUX_RUNTIME_SHA256S:
            if not all((output / model_name / "complete.json").exists() for model_name in cfg["backbones"]):
                raise ValueError("This unfinished run uses earlier training code. Keep its original notebook to resume it; do not mix implementations in one run.")
            expected_signature["runtime_sha256"] = saved_signature["runtime_sha256"]
        if saved_signature != expected_signature:
            raise ValueError("Existing outputs belong to a different configuration. Use a new output version/folder.")
        signature = saved_signature
    aux_json(identity_path, signature)
    aux_json(output / "run_environment.json", {"device": str(device),
        "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else None,
        "notebook_code_sha256": AUX_RUNTIME_SHA256})
    for name in ("train", "val", "test", "audit"):
        frames[name][["uniqueID", "split", "narrative_group"]].to_csv(output / f"{name}_ids.csv", index=False)
    print("Partitions:", identity["split_counts"], "| target:", cfg["target"], "| anchored:", cfg["aspects"], "| free:", cfg["free_factors"], "| residual CLS:", cfg["residual_cls"])
    train_labels = aux_task_labels(frames["train"], cfg)
    criteria, class_weights = aux_criteria(train_labels, device)
    aux_json(output / "train_class_weights.json", class_weights)
    return output, device, plan, criteria, class_weights


## 3.2 Data splits \& labels

In [15]:
train_labels = aux_task_labels(frames["train"], AUX_CONFIG)
val_labels = aux_task_labels(frames["val"], AUX_CONFIG)
test_labels = aux_task_labels(frames["test"], AUX_CONFIG)

output, device, plan, criteria, class_weights = prepare_experiment(
    PROJECT_ROOT, AUX_CONFIG, frames, identity
)
print("Parameter search:")
print(pd.DataFrame(plan).to_string(index=False))
print("Class weights from training data:", class_weights)


Partitions: {'train': 60950, 'val': 20256, 'test': 20298, 'audit': 557} | target: holistic | anchored: ['efficacy', 'safety', 'burden', 'cost'] | free: 0 | residual CLS: False
Parameter search:
      lr  epochs  dropout  seed
0.000012       6 0.085351  2025
0.000011       7 0.097059  2026
0.000013       9 0.195692  2027
0.000011       9 0.200246  2028
0.000016       4 0.094601  2029
0.000018       5 0.000793  2030
0.000014       9 0.183296  2031
0.000011       7 0.062150  2032
0.000021       6 0.155208  2033
0.000030       9 0.236536  2034
Class weights from training data: {'efficacy': [1.7274608612060547, 1.9261155128479004, 0.5257800221443176], 'safety': [0.653416097164154, 0.9769037365913391, 2.242457628250122], 'burden': [0.8348743319511414, 0.6606616377830505, 3.4652340412139893], 'cost': [7.353118419647217, 0.3579587936401367, 14.207459449768066], 'overall': [1.098297119140625, 1.206214189529419, 6.200407028198242, 0.6612063646316528, 0.6302021145820618]}


## 3.3 Semi-confirmatory model components

Shared encoder → one token-attention factor per anchored aspect and per free factor → aspect heads on the anchored factors only → review-specific factor fusion (optionally with CLS) → holistic head. Special-token and padding positions are excluded from every factor's attention.

* **Anchored factors** are `05_`'s aspect representations: each attends over the encoder's token states and predicts three classes from `h_k`, supervised by the aspect label.
* **Free factors** are the same attention heads without an aspect head. They are trained only by the holistic loss and by the orthogonality penalty, so they must find token patterns that help the holistic prediction and that the anchored factors do not already attend to.
* **Orthogonality penalty.** For each review, the squared cosine similarity between the attention distributions of factor pairs that involve at least one free factor, averaged over pairs and reviews. Cosine rather than raw dot product, so two identical diffuse attentions count as maximally overlapping and the penalty cannot be satisfied by spreading attention thin. Pairs of anchored factors are not penalized; their labels already separate them.
* **Fusion and holistic head.** `β = softmax_k(s(h_k))` over all factors, `h = Σ_k β_k h_k`; the holistic head reads `h`, or `[CLS ; h]` when `RESIDUAL_CLS` is True. `β` is saved for every factor.

Loss: overall weighted cross-entropy + `AUXILIARY_STRENGTH` × mean anchored-aspect weighted cross-entropy + `ORTHOGONALITY` × penalty, with TRAIN-only class weights. Selection uses validation holistic weighted F1, as in `04_`/`05_`.

**Reading the results.** Four anchored + zero free is the confirmatory model and must reproduce `05_` `representation_only`. Adding free factors and comparing holistic scores measures whether unconstrained attention pathways carry holistic information beyond the four aspect-supervised pathways. Because anchored representations are contextual vectors rather than pure concept values, a gain does not by itself show that separate latent aspects exist; the free factors are **unconstrained residual representation factors** until interpreted. The factor-loading table (`*_factor_loadings.csv`: attention mass on each aspect's GPT evidence spans divided by chance) says whether a free factor is enriched on the selected evidence of a known aspect. A lift near 1 everywhere means only that the factor is not enriched on the available selected quotations; it may still carry unselected evidence for a known aspect, global sentiment, diffuse attention, or something new, and the audit top tokens and spans are where to look. Free factors are exchangeable across runs, so interpret a factor only after it recurs across independent seeds at the selected hyperparameters (match factors by loading profile). `FREE_FACTORS = 4` and `ORTHOGONALITY = 0.1` are pre-specified; a sensitivity sweep over m ∈ {2, 4, 8} and λ ∈ {0, 0.1} on one backbone belongs in the supplement, not in model selection. The `suppress` argument of the model's forward pass supports a post-hoc intervention: remove factors at inference, renormalize the remaining fusion weights, and measure the change in holistic metrics. With zero anchored factors the same notebook is the exploratory model of this family.


In [16]:

def aux_verify_completed_run(model_output):
    """A completion shortcut requires a complete, unchanged set of run artifacts."""
    model_output = Path(model_output)
    complete = json.loads((model_output / "complete.json").read_text())
    artifacts = complete.get("artifacts")
    if not isinstance(artifacts, dict) or not artifacts:
        raise ValueError("Completed run has an empty or invalid artifact manifest.")
    model_name = model_output.name
    required = {"best_model.pth", "metrics.csv", "selected_parameters.json"}
    required.update(f"{model_name}_{part}_predictions.csv" for part in ("val", "test", "audit"))
    required.update(f"{model_name}_{part}_factor_loadings.csv" for part in ("val", "test", "audit"))
    missing = required - set(artifacts)
    if missing:
        raise ValueError(f"Completed run manifest omits required artifacts: {sorted(missing)}")
    base = model_output.resolve()
    for filename, digest in artifacts.items():
        path = model_output / filename
        if Path(filename).is_absolute() or base not in path.resolve().parents:
            raise ValueError(f"Completed artifact escapes its run directory: {filename}")
        if not path.is_file() or aux_hash(path) != digest:
            raise ValueError(f"Completed artifact changed: {model_name}/{filename}")
    verify_completed_numerics(model_output)
    return complete

def verify_completed_numerics(model_output):
    for prediction_path in model_output.glob("*_predictions.csv"):
        saved = pd.read_csv(prediction_path)
        for task in ["overall", *AUX_ASPECTS]:
            names = AUX_OVERALL if task == "overall" else AUX_STATES
            columns = [f"{task}_p_{name}" for name in names]
            if all(c in saved for c in columns):
                require_probabilities(saved[columns].to_numpy(), len(names))
        fusion_columns = [c for c in saved.columns if c.startswith("fusion_w_")]
        if fusion_columns:
            require_probabilities(saved[fusion_columns].to_numpy(), len(fusion_columns))
    payload = torch.load(model_output / "best_model.pth", map_location="cpu", weights_only=True)
    for name, value in payload["state_dict"].items():
        if value.is_floating_point():
            require_finite_tensor(value, f"saved model state {name}")

class AuxiliaryDataset(Dataset):
    """All labels are supervision only; model.forward accepts text tensors only. Evidence-token masks are for evaluation."""
    def __init__(self, encoded, labels, evidence_tokens=None):
        if not {"input_ids", "attention_mask", "special_tokens_mask", "offset_mapping"}.issubset(encoded):
            raise ValueError("Tokenization must include special-token masks and offsets.")
        self.encoded = encoded
        self.labels = {k: torch.as_tensor(v, dtype=torch.long) for k, v in labels.items()}
        self.evidence_tokens = evidence_tokens

    def __len__(self):
        return len(self.encoded["input_ids"])

    def __getitem__(self, index):
        item = {"input_ids": self.encoded["input_ids"][index],
                "attention_mask": self.encoded["attention_mask"][index],
                "special_tokens_mask": self.encoded["special_tokens_mask"][index],
                "offset_mapping": self.encoded["offset_mapping"][index],
                "targets": {k: v[index] for k, v in self.labels.items()}}
        if self.evidence_tokens is not None:
            item["evidence_tokens"] = self.evidence_tokens[index]
        return item

class AspectTokenAttention(nn.Module):
    """One additive attention head per factor over the encoder token states."""
    def __init__(self, hidden_size, n_factors, attention_size):
        super().__init__()
        self.project = nn.Linear(hidden_size, attention_size)
        self.query = nn.Parameter(torch.empty(n_factors, attention_size))
        nn.init.normal_(self.query, std=0.02)

    def forward(self, hidden, attention_mask):
        if attention_mask.ndim != 2 or attention_mask.shape != hidden.shape[:2] or not attention_mask.bool().any(dim=1).all():
            raise ValueError("Every review needs at least one eligible content token.")
        keys = torch.tanh(self.project(hidden))                                   # (B, T, A)
        with torch.autocast(device_type=hidden.device.type, enabled=False):      # scores and softmax in float32
            scores = torch.einsum("bta,ka->bkt", keys.float(), self.query.float())  # (B, F, T)
            scores = scores.masked_fill(~attention_mask.bool().unsqueeze(1), torch.finfo(scores.dtype).min)
            alpha = scores.softmax(-1)
            pooled = torch.einsum("bkt,btd->bkd", alpha, hidden.float())          # (B, F, d)
        return pooled, alpha

class SemiConfirmatoryClassifier(nn.Module):
    """Shared encoder -> anchored + free attention factors -> aspect heads (anchored) -> fusion (+ CLS) -> holistic head."""
    def __init__(self, base, aspects, free_factors, dropout=0.1, attention_size=256, residual_cls=False):
        super().__init__()
        self.base = base
        self.aspects = tuple(aspects)
        self.free_factors = int(free_factors)
        self.residual_cls = bool(residual_cls)
        n_factors = len(self.aspects) + self.free_factors
        if n_factors < 1:
            raise ValueError("At least one factor is required.")
        hidden_size = base.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.token_attention = AspectTokenAttention(hidden_size, n_factors, attention_size)
        self.aspect_heads = nn.ModuleDict({a: nn.Linear(hidden_size, len(AUX_STATES)) for a in self.aspects})
        self.fusion_scorer = nn.Sequential(nn.Linear(hidden_size, attention_size), nn.Tanh(), nn.Linear(attention_size, 1))
        self.primary_head = nn.Linear(hidden_size * (2 if self.residual_cls else 1), len(AUX_OVERALL))
        # Pairs that carry the orthogonality penalty: every pair involving at least one free factor.
        pair_mask = torch.zeros(n_factors, n_factors, dtype=torch.bool)
        for i in range(n_factors):
            for j in range(i + 1, n_factors):
                if i >= len(self.aspects) or j >= len(self.aspects):
                    pair_mask[i, j] = True
        self.register_buffer("pair_mask", pair_mask)

    def forward(self, input_ids, attention_mask, special_tokens_mask=None, return_extras=False, suppress=None):
        """`suppress`: optional list of factor indices whose fusion weight is set to zero at inference
        (post-hoc intervention analysis only; the remaining weights are renormalized)."""
        if special_tokens_mask is None or special_tokens_mask.shape != input_ids.shape:
            raise ValueError("An aligned special_tokens_mask is required.")
        eligible = attention_mask.bool() & ~special_tokens_mask.bool()
        hidden = self.base(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        pooled, alpha = self.token_attention(hidden, eligible)                    # (B, F, d), (B, F, T)
        pooled = self.dropout(pooled)
        result = {a: head(pooled[:, k]) for k, (a, head) in enumerate(self.aspect_heads.items())}
        with torch.autocast(device_type=hidden.device.type, enabled=False):
            fusion = self.fusion_scorer(pooled.float()).squeeze(-1).softmax(-1)  # (B, F)
            if suppress:
                keep = torch.ones(fusion.shape[1], device=fusion.device)
                keep[list(suppress)] = 0.0
                if keep.sum() == 0:
                    raise ValueError("At least one factor must remain.")
                fusion = fusion * keep
                fusion = fusion / fusion.sum(-1, keepdim=True).clamp_min(1e-12)
            summary = (fusion.unsqueeze(-1) * pooled.float()).sum(1)              # (B, d)
            # Squared cosine overlap between attention distributions: two identical diffuse
            # attentions score 1, so the penalty cannot be satisfied by spreading attention out.
            unit = alpha / alpha.norm(dim=-1, keepdim=True).clamp_min(1e-12)
            overlap = torch.einsum("bit,bjt->bij", unit, unit)                   # (B, F, F)
            # No penalized pairs (confirmatory design) -> no penalty term at all.
            penalty = overlap[:, self.pair_mask].square().mean() if bool(self.pair_mask.any()) else None
        features = torch.cat([self.dropout(hidden[:, 0, :]).float(), summary], dim=-1) if self.residual_cls else summary
        result["overall"] = self.primary_head(features)
        if return_extras:
            return result, {"fusion": fusion, "token_attention": alpha, "penalty": penalty}
        return result

def aux_fit(model, train_loader, val_loader, criteria, hp, cfg, device):
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=hp["lr"], weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=hp["lr"],
        total_steps=hp["epochs"] * len(train_loader), pct_start=0.1)
    use_amp = device.type == "cuda"
    dtype = torch.bfloat16 if use_amp and torch.cuda.is_bf16_supported() else torch.float16
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp and dtype == torch.float16)
    best, state, best_epoch, stalled = -1.0, None, -1, 0
    history = []
    for epoch in range(1, hp["epochs"] + 1):
        model.train()
        losses, per_task = [], {}
        for batch in train_loader:
            targets = {k: v.to(device) for k, v in batch["targets"].items()}
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=dtype, enabled=use_amp):
                logits, extras = model(batch["input_ids"].to(device), batch["attention_mask"].to(device),
                    special_tokens_mask=batch["special_tokens_mask"].to(device), return_extras=True)
                loss, terms = aux_loss(logits, targets, criteria, cfg["aux_strength"], cfg["orthogonality"], extras["penalty"])
            checked_optimizer_step(loss, model, optimizer, scaler, 1.0, scheduler)
            losses.append(loss.item())
            for task, value in terms.items():
                per_task.setdefault(task, []).append(value.item())
        probs, labels, _ = aux_predict(model, val_loader, device)
        score = aux_selection(probs, labels)
        row = {"epoch": epoch, "train_loss": float(np.mean(losses)), "selection_f1": score,
               "task_losses": {k: float(np.mean(v)) for k, v in per_task.items()},
               "validation": {k: aux_metrics(labels[k], p.argmax(1), k == "overall") for k, p in probs.items()}}
        history.append(row)
        print(f"Epoch {epoch}/{hp['epochs']} | loss={row['train_loss']:.4f} | validation selection F1={score:.4f}", flush=True)
        if score > best:
            require_finite_model(model)
            best, best_epoch, stalled = score, epoch, 0
            state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            stalled += 1
            if stalled >= cfg["patience"]:
                break
    if state is None:
        raise RuntimeError("No valid checkpoint.")
    model.load_state_dict(state)
    return best, best_epoch, history

def prepare_model(model_name, frames, cfg, output, device):
    """Tokenize this model's training/validation text, or verify its completed run."""
    from transformers import AutoTokenizer
    if model_name not in cfg["backbones"]:
        print(model_name, "is not selected in BACKBONES; skipping.")
        return None
    model_output = output / model_name
    model_output.mkdir(exist_ok=True)
    done = model_output / "complete.json"
    run = {"name": model_name, "output": model_output, "done": done,
           "complete": done.exists()}
    if done.exists():
        aux_verify_completed_run(model_output)
        print(model_name, "already complete and verified; skipping.")
        return run
    tokenizer = AutoTokenizer.from_pretrained(AUX_BACKBONES[model_name], use_fast=True)
    datasets = {}
    # Test/audit predictions are made after selecting the model on validation data.
    for partition in ("train", "val"):
        encoded = tokenizer(frames[partition].model_text.tolist(), truncation=True,
            padding="max_length", max_length=cfg["max_length"], return_tensors="pt",
            return_special_tokens_mask=True, return_offsets_mapping=True)
        datasets[partition] = AuxiliaryDataset(encoded, aux_task_labels(frames[partition], cfg))
    loader_args = {"batch_size": cfg["batch_size"], "num_workers": cfg["num_workers"],
                   "pin_memory": device.type == "cuda"}
    val_loader = DataLoader(datasets["val"], shuffle=False,
        generator=torch.Generator().manual_seed(cfg["seed"] + 100_000), **loader_args)
    run.update(tokenizer=tokenizer, datasets=datasets, loader_args=loader_args,
               val_loader=val_loader)
    return run

def evaluate_saved_model(run, frames, cfg, best_record, device):
    """Load the selected model, evaluate each partition (with evidence-token masks) and save the outputs."""
    from transformers import AutoModel
    if run is None:
        return pd.DataFrame(columns=["model", "task", "partition"])
    if run["complete"]:
        aux_verify_completed_run(run["output"])
        return pd.read_csv(run["output"] / "metrics.csv")
    model_name, model_output, done = run["name"], run["output"], run["done"]
    tokenizer, datasets = run["tokenizer"], run["datasets"]
    loader_args = run["loader_args"]
    payload = torch.load(model_output / "best_model.pth", map_location="cpu", weights_only=True)
    if payload["record"] != best_record:
        raise ValueError("Selected checkpoint does not match completed trial records.")
    model = SemiConfirmatoryClassifier(AutoModel.from_pretrained(AUX_BACKBONES[model_name]),
        cfg["aspects"], cfg["free_factors"], dropout=best_record["hp"]["dropout"],
        attention_size=cfg["attention_size"], residual_cls=cfg["residual_cls"])
    model.load_state_dict(payload["state_dict"], strict=True)
    require_finite_model(model)
    model.to(device)
    rows = []
    for partition in ("val", "test", "audit"):
        if partition not in datasets:
            encoded = tokenizer(frames[partition].model_text.tolist(), truncation=True,
                padding="max_length", max_length=cfg["max_length"], return_tensors="pt",
                return_special_tokens_mask=True, return_offsets_mapping=True)
            datasets[partition] = AuxiliaryDataset(encoded, aux_task_labels(frames[partition], cfg))
        # Evidence-token masks are attached for evaluation only; they never enter training or selection.
        datasets[partition].evidence_tokens = aux_evidence_tokens(datasets[partition].encoded, frames[partition])
        loader = DataLoader(datasets[partition], shuffle=False,
            generator=torch.Generator().manual_seed(cfg["seed"] + 200_000), **loader_args)
        rows += aux_export(model, loader, frames[partition], cfg, model_name, partition, model_output, device, tokenizer=tokenizer)
    pd.DataFrame(rows).to_csv(model_output / "metrics.csv", index=False)
    tokenizer.save_pretrained(model_output / "tokenizer")
    model.base.config.save_pretrained(model_output / "encoder_config")
    aux_json(model_output / "selected_parameters.json", best_record)
    print(pd.DataFrame(rows).query("partition == 'test'").to_string(index=False))
    artifacts = {p.relative_to(model_output).as_posix(): aux_hash(p)
                 for p in model_output.rglob("*") if p.is_file() and p != done}
    aux_json(done, {"artifacts": artifacts})
    del model, loader
    run.pop("datasets", None)
    run.pop("val_loader", None)
    run["complete"] = True
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return pd.DataFrame(rows)


## 3.4 ALBERT: Semi-confirmatory training

The following three steps prepare text, search the existing parameter range and evaluate the selected model. Completed models and trials are reused as before; restarting Colab alone does not force retraining.


In [17]:
print("\n" + "=" * 60)
print('ALBERT' + " — DrugReview semi-confirmatory training")
print("=" * 60)



ALBERT — DrugReview semi-confirmatory training


### 3.4.1 Pre-tokenize

In [18]:
albert_run = prepare_model('albert', frames, AUX_CONFIG, output, device)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/760k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

### 3.4.2 Hyperparameter tuning

In [19]:
best_record_albert = None
if albert_run is not None and not albert_run["complete"]:
    for index, hp in enumerate(plan):
        trial_path = albert_run["output"] / f"trial_{index + 1}.json"
        # A saved completed trial can be reused on interruption; the selected checkpoint is always updated first.
        if trial_path.exists():
            record = json.loads(trial_path.read_text())
            if record["hp"] != hp:
                raise ValueError("Saved trial plan differs.")
            if best_record_albert is None or record["score"] > best_record_albert["score"]:
                best_record_albert = record
            continue
        aux_seed(hp["seed"])
        model = SemiConfirmatoryClassifier(AutoModel.from_pretrained(AUX_BACKBONES['albert']),
            AUX_CONFIG["aspects"], AUX_CONFIG["free_factors"], dropout=hp["dropout"],
            attention_size=AUX_CONFIG["attention_size"], residual_cls=AUX_CONFIG["residual_cls"])
        # Reset after head construction: identical data order and dropout RNG across paired conditions.
        aux_seed(hp["seed"])
        train_loader = DataLoader(albert_run["datasets"]["train"], shuffle=True,
            generator=torch.Generator().manual_seed(hp["seed"]), worker_init_fn=aux_worker_seed, **albert_run["loader_args"])
        print(f"albert trial {index + 1}/{len(plan)}", hp, flush=True)
        score, epoch, history = aux_fit(model, train_loader, albert_run["val_loader"], criteria, hp, AUX_CONFIG, device)
        record = {"trial": index + 1, "hp": hp, "score": score, "selected_epoch": epoch}
        if best_record_albert is None or score > best_record_albert["score"]:
            best_record_albert = record
            tmp = albert_run["output"] / "best_model.tmp"
            torch.save({"state_dict": {k: v.cpu() for k, v in model.state_dict().items()}, "record": record}, tmp)
            tmp.replace(albert_run["output"] / "best_model.pth")
        aux_json(albert_run["output"] / f"trial_{index + 1}_history.json", history)
        aux_json(trial_path, record)
        del model, train_loader
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()


model.safetensors:   0%|          | 0.00/47.4M [00:00<?, ?B/s]

albert trial 1/10 {'lr': 1.160497696239576e-05, 'epochs': 6, 'dropout': 0.08535068283342875, 'seed': 2025}


/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:900.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 1/6 | loss=2.0354 | validation selection F1=0.6904
Epoch 2/6 | loss=1.3667 | validation selection F1=0.7133
Epoch 3/6 | loss=1.1593 | validation selection F1=0.7506
Epoch 4/6 | loss=1.0059 | validation selection F1=0.7610
Epoch 5/6 | loss=0.8989 | validation selection F1=0.7673
Epoch 6/6 | loss=0.8432 | validation selection F1=0.7682
albert trial 2/10 {'lr': 1.1229524450717097e-05, 'epochs': 7, 'dropout': 0.09705888652849565, 'seed': 2026}
Epoch 1/7 | loss=2.1377 | validation selection F1=0.6823
Epoch 2/7 | loss=1.4153 | validation selection F1=0.7175
Epoch 3/7 | loss=1.1849 | validation selection F1=0.7452
Epoch 4/7 | loss=1.0235 | validation selection F1=0.7601
Epoch 5/7 | loss=0.9012 | validation selection F1=0.7658
Epoch 6/7 | loss=0.8155 | validation selection F1=0.7671
Epoch 7/7 | loss=0.7769 | validation selection F1=0.7682
albert trial 3/10 {'lr': 1.3271033162649868e-05, 'epochs': 9, 'dropout': 0.195692122900145, 'seed': 2027}
Epoch 1/9 | loss=2.2032 | validation selectio

### 3.4.3 Model evaluation

In [20]:
albert_results = evaluate_saved_model(
    albert_run, frames, AUX_CONFIG, best_record_albert, device
)
print(albert_results.query("partition == 'test'").to_string(index=False))


Evidence quotes: {'total': 81708, 'unmatched': 2593, 'ambiguous': 53, 'truncated': 104, 'used': 78958} (only unique, fully visible quotes define evidence tokens)
Evidence quotes: {'total': 82160, 'unmatched': 2591, 'ambiguous': 54, 'truncated': 175, 'used': 79340} (only unique, fully visible quotes define evidence tokens)
Evidence quotes: {'total': 2424, 'unmatched': 67, 'ambiguous': 0, 'truncated': 0, 'used': 2357} (only unique, fully visible quotes define evidence tokens)
 model         variant     task partition reference_source  human_agreement     n  accuracy  weighted_f1  macro_f1      mae      qwk
albert 4anchored_0free efficacy      test      GPT-4o-mini            False 20298  0.755050     0.773750  0.703879      NaN      NaN
albert 4anchored_0free   safety      test      GPT-4o-mini            False 20298  0.782934     0.783484  0.769886      NaN      NaN
albert 4anchored_0free   burden      test      GPT-4o-mini            False 20298  0.669278     0.679053  0.621432      Na

## 3.5 BioBERT: Semi-confirmatory training

The following three steps prepare text, search the existing parameter range and evaluate the selected model. Completed models and trials are reused as before; restarting Colab alone does not force retraining.


In [21]:
print("\n" + "=" * 60)
print('BioBERT' + " — DrugReview semi-confirmatory training")
print("=" * 60)



BioBERT — DrugReview semi-confirmatory training


### 3.5.1 Pre-tokenize

In [22]:
biobert_run = prepare_model('biobert', frames, AUX_CONFIG, output, device)


config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

### 3.5.2 Hyperparameter tuning

In [23]:
best_record_biobert = None
if biobert_run is not None and not biobert_run["complete"]:
    for index, hp in enumerate(plan):
        trial_path = biobert_run["output"] / f"trial_{index + 1}.json"
        # A saved completed trial can be reused on interruption; the selected checkpoint is always updated first.
        if trial_path.exists():
            record = json.loads(trial_path.read_text())
            if record["hp"] != hp:
                raise ValueError("Saved trial plan differs.")
            if best_record_biobert is None or record["score"] > best_record_biobert["score"]:
                best_record_biobert = record
            continue
        aux_seed(hp["seed"])
        model = SemiConfirmatoryClassifier(AutoModel.from_pretrained(AUX_BACKBONES['biobert']),
            AUX_CONFIG["aspects"], AUX_CONFIG["free_factors"], dropout=hp["dropout"],
            attention_size=AUX_CONFIG["attention_size"], residual_cls=AUX_CONFIG["residual_cls"])
        # Reset after head construction: identical data order and dropout RNG across paired conditions.
        aux_seed(hp["seed"])
        train_loader = DataLoader(biobert_run["datasets"]["train"], shuffle=True,
            generator=torch.Generator().manual_seed(hp["seed"]), worker_init_fn=aux_worker_seed, **biobert_run["loader_args"])
        print(f"biobert trial {index + 1}/{len(plan)}", hp, flush=True)
        score, epoch, history = aux_fit(model, train_loader, biobert_run["val_loader"], criteria, hp, AUX_CONFIG, device)
        record = {"trial": index + 1, "hp": hp, "score": score, "selected_epoch": epoch}
        if best_record_biobert is None or score > best_record_biobert["score"]:
            best_record_biobert = record
            tmp = biobert_run["output"] / "best_model.tmp"
            torch.save({"state_dict": {k: v.cpu() for k, v in model.state_dict().items()}, "record": record}, tmp)
            tmp.replace(biobert_run["output"] / "best_model.pth")
        aux_json(biobert_run["output"] / f"trial_{index + 1}_history.json", history)
        aux_json(trial_path, record)
        del model, train_loader
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()


pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

biobert trial 1/10 {'lr': 1.160497696239576e-05, 'epochs': 6, 'dropout': 0.08535068283342875, 'seed': 2025}


/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:900.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Epoch 1/6 | loss=2.2186 | validation selection F1=0.6495
Epoch 2/6 | loss=1.5414 | validation selection F1=0.7100
Epoch 3/6 | loss=1.3083 | validation selection F1=0.7208
Epoch 4/6 | loss=1.1652 | validation selection F1=0.7402
Epoch 5/6 | loss=1.0806 | validation selection F1=0.7394
Epoch 6/6 | loss=1.0441 | validation selection F1=0.7408
biobert trial 2/10 {'lr': 1.1229524450717097e-05, 'epochs': 7, 'dropout': 0.09705888652849565, 'seed': 2026}
Epoch 1/7 | loss=2.2160 | validation selection F1=0.6483
Epoch 2/7 | loss=1.5276 | validation selection F1=0.7056
Epoch 3/7 | loss=1.3062 | validation selection F1=0.7374
Epoch 4/7 | loss=1.1612 | validation selection F1=0.7391
Epoch 5/7 | loss=1.0617 | validation selection F1=0.7430
Epoch 6/7 | loss=1.0007 | validation selection F1=0.7448
Epoch 7/7 | loss=0.9741 | validation selection F1=0.7471
biobert trial 3/10 {'lr': 1.3271033162649868e-05, 'epochs': 9, 'dropout': 0.195692122900145, 'seed': 2027}
Epoch 1/9 | loss=2.2988 | validation select

### 3.5.3 Model Evaluation

In [24]:
biobert_results = evaluate_saved_model(
    biobert_run, frames, AUX_CONFIG, best_record_biobert, device
)
print(biobert_results.query("partition == 'test'").to_string(index=False))


Evidence quotes: {'total': 81708, 'unmatched': 2593, 'ambiguous': 53, 'truncated': 113, 'used': 78949} (only unique, fully visible quotes define evidence tokens)
Evidence quotes: {'total': 82160, 'unmatched': 2591, 'ambiguous': 54, 'truncated': 181, 'used': 79334} (only unique, fully visible quotes define evidence tokens)
Evidence quotes: {'total': 2424, 'unmatched': 67, 'ambiguous': 0, 'truncated': 0, 'used': 2357} (only unique, fully visible quotes define evidence tokens)
  model         variant     task partition reference_source  human_agreement     n  accuracy  weighted_f1  macro_f1      mae      qwk
biobert 4anchored_0free efficacy      test      GPT-4o-mini            False 20298  0.760370     0.776488  0.703993      NaN      NaN
biobert 4anchored_0free   safety      test      GPT-4o-mini            False 20298  0.786186     0.786074  0.770069      NaN      NaN
biobert 4anchored_0free   burden      test      GPT-4o-mini            False 20298  0.652232     0.666768  0.608915    

## 3.6 Save Prediction

The evaluation cells save per-review predictions and model files in the `albert/` and `biobert/` output folders. Prediction files carry the fusion weight of every factor (`fusion_w_<aspect>` and `fusion_w_free_<i>`); the audit-partition file adds each factor's most attended tokens and character spans; `<model>_<partition>_factor_loadings.csv` gives each factor's attention mass on the GPT evidence spans of each aspect relative to chance. This last cell saves a combined metrics summary and prints the test-set loadings. Test and human-audit predictions remain separate.


In [25]:
all_model_metrics = pd.concat([albert_results, biobert_results], ignore_index=True)
summary_path = output / "all_models_metrics.csv"
all_model_metrics.to_csv(summary_path, index=False)
print(f"Saved model summary → {summary_path}")
for model_name in AUX_CONFIG["backbones"]:
    for partition in ("val", "test", "audit"):
        path = output / model_name / f"{model_name}_{partition}_predictions.csv"
        print(f"{model_name} {partition} predictions → {path}")
    fusion_path = output / model_name / f"{model_name}_fusion_by_holistic_class.csv"
    if fusion_path.exists():
        print(f"{model_name} mean fusion weights by holistic class → {fusion_path}")
    loading_path = output / model_name / f"{model_name}_test_factor_loadings.csv"
    if loading_path.exists():
        loadings = pd.read_csv(loading_path)
        print(f"{model_name} factor loadings (test, lift of attention on each aspect's evidence over chance):")
        print(loadings.pivot(index="factor", columns="aspect", values="lift").reindex(FACTOR_NAMES).round(2).to_string())
print(all_model_metrics.query("partition == 'test'").to_string(index=False))


Saved model summary → /content/drive/MyDrive/NLP_Projects/03_DrugReview/06_Semi_confirmatory/outputs/semi_4anchored_0free_v2/all_models_metrics.csv
albert val predictions → /content/drive/MyDrive/NLP_Projects/03_DrugReview/06_Semi_confirmatory/outputs/semi_4anchored_0free_v2/albert/albert_val_predictions.csv
albert test predictions → /content/drive/MyDrive/NLP_Projects/03_DrugReview/06_Semi_confirmatory/outputs/semi_4anchored_0free_v2/albert/albert_test_predictions.csv
albert audit predictions → /content/drive/MyDrive/NLP_Projects/03_DrugReview/06_Semi_confirmatory/outputs/semi_4anchored_0free_v2/albert/albert_audit_predictions.csv
albert mean fusion weights by holistic class → /content/drive/MyDrive/NLP_Projects/03_DrugReview/06_Semi_confirmatory/outputs/semi_4anchored_0free_v2/albert/albert_fusion_by_holistic_class.csv
albert factor loadings (test, lift of attention on each aspect's evidence over chance):
aspect    burden  cost  efficacy  safety
factor                                